![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Recommendation Systems: Personalized Content Filtering with RedisVL

<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/recommendation-systems/04_personalized_content_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook follows [00_content_filtering.ipynb](00_content_filtering.ipynb), using the same IMDB movie dataset and sentence-transformer model. It runs independently, so you don't need to run the earlier notebook first.

We'll add lightweight personalization with two functions:

1. When a user clicks or views a movie, read its existing embedding from Redis and update a session vector using an exponentially weighted moving average (EWMA).
2. At search time, blend the query embedding with that session vector and pass the result into a standard RedisVL `VectorQuery`.

The session vector lives in a plain Redis string key, `session:{user_id}:vec`, with a 30-minute TTL. Each interaction refreshes the TTL; searches do not. After it expires, searches fall back to the query embedding alone. No additional model training is needed.

## Environment Setup

In [ ]:
%pip install -q redis "redisvl>=0.5.1" sentence_transformers pandas requests numpy

### Start Redis

Use Redis 8 or a Redis Stack instance with JSON and vector search support. The following cell installs Redis locally in Google Colab; skip it when using an existing instance.

In [ ]:
%%sh
# NBVAL_SKIP
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update  > /dev/null 2>&1
sudo apt-get install -y redis-server  > /dev/null 2>&1
sudo redis-server /etc/redis/redis.conf --daemonize yes

#### Other ways to get Redis
There are many ways to get the necessary redis-stack instance running
1. On cloud, deploy a [FREE instance of Redis in the cloud](https://redis.io/try-free/). Or, if you have your
own version of Redis Enterprise running, that works too!
2. Per OS, [see the docs](https://redis.io/docs/latest/operate/oss_and_stack/install/install-stack/)
3. With docker: `docker run -d --name redis -p 6379:6379 redis:8`

### Define the Redis connection

By default we connect to `redis://localhost:6379`. Set `REDIS_URL` to use your own instance, including credentials and a `rediss://` scheme if TLS is required. Keep `decode_responses=False` because session vectors are binary float32 data.

In [ ]:
import ast
import os
import pickle
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import requests
from redis import Redis
from redisvl.index import SearchIndex
from redisvl.query import VectorQuery
from redisvl.schema import IndexSchema
from redisvl.utils.vectorize import HFTextVectorizer

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
client = Redis.from_url(REDIS_URL, decode_responses=False)
client.ping()

## Prepare the dataset

Reuse the dataset and cached embeddings from the content filtering notebook. If the local files are missing, download the same prepared files. Only load pickle files from a trusted source.

In [ ]:
data_dir = Path("datasets/content_filtering")
data_dir.mkdir(parents=True, exist_ok=True)
base_url = "https://redis-ai-resources.s3.us-east-2.amazonaws.com/recommenders/datasets/content-filtering"

for filename in ["25k_imdb_movie_dataset.csv", "text_embeddings.pkl"]:
    path = data_dir / filename
    if not path.exists():
        response = requests.get(f"{base_url}/{filename}", timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)

df = pd.read_csv(data_dir / "25k_imdb_movie_dataset.csv")
df.head()

Remove unused fields, parse list columns, convert release years to timestamps, and exclude movies without a usable overview.


In [ ]:
import datetime
roman_numerals = ['0','(I)','(II)','(III)','(IV)', '(V)', '(VI)', '(VII)', '(VIII)', '(IX)', '(XI)', '(XII)', '(XVI)', '(XIV)', '(XXXIII)', '(XVIII)', '(XIX)', '(XXVII)']

def replace_year(x):
    if x in roman_numerals:
        return datetime.datetime(1998, 1, 1).timestamp()
    else:
        return datetime.datetime(int(x), 1, 1).timestamp()

df.drop(columns=['runtime', 'writer', 'path'], inplace=True)
df['year'] = df['year'].apply(replace_year)             # replace roman numerals with average year as a timestamp
df['genres'] = df['genres'].apply(ast.literal_eval)     # convert string representation of list to list
df['keywords'] = df['keywords'].apply(ast.literal_eval) # convert string representation of list to list
df['cast'] = df['cast'].apply(ast.literal_eval)         # convert string representation of list to list
df = df[~df['overview'].isnull()]                       # drop rows with missing overviews
df = df[~df['overview'].isin(['none'])]                 # drop rows with 'none' as the overview

# make sure we've filled all missing values
df.isnull().sum()

In [ ]:
# add a column to the dataframe with all the text we want to embed
df["full_text"] = df["title"] + ". " + df["overview"] + " " + df['keywords'].apply(lambda x: ', '.join(x))
df["full_text"][0]

## Load movie embeddings and initialize the query model

The cached movie vectors were generated with `sentence-transformers/paraphrase-MiniLM-L6-v2`. Use that exact model for query embeddings so movie, session, and query vectors share the same 384-dimensional space. Loading the cached vectors avoids embedding the entire dataset again.

In [ ]:
vectorizer = HFTextVectorizer(model="sentence-transformers/paraphrase-MiniLM-L6-v2")

with (data_dir / "text_embeddings.pkl").open("rb") as vector_file:
    df["embedding"] = pickle.load(vector_file)

assert df["embedding"].notna().all(), "Every movie must have a cached embedding."
assert all(len(vector) == vectorizer.dims for vector in df["embedding"])

## Define the search schema and load movies

Use the same JSON storage and cosine vector search as the content filtering notebook. This example uses a separate `movies_personalized_recommendation` index and `personalized_movie:` key prefix so it can be run alongside that notebook. Session keys are outside the movie index.

Movie keys use the dataframe row IDs, making reloading the same dataset safe without duplicating movies.

In [ ]:
# define our movie schema
movie_schema = IndexSchema.from_dict(
    {
        'index': {
            'name': 'movies_personalized_recommendation',
            'prefix': 'personalized_movie:',
            'storage_type': 'json'
        },
        'fields': [
            {'name': 'title', 'type': 'text'},
            {'name': 'rating', 'type': 'numeric'},
            {'name': 'rating_count', 'type': 'numeric'},
            {'name': 'genres', 'type': 'tag'},
            {'name': 'overview', 'type': 'text'},
            {'name': 'keywords', 'type': 'tag'},
            {'name': 'cast', 'type': 'tag'},
            {'name': 'year', 'type': 'numeric'},
            {'name': 'full_text', 'type': 'text'},
            {
                'name': 'embedding',
                'type': 'vector',
                'attrs': {
                    'dims': 384,
                    'algorithm': 'flat',
                    'datatype': 'float32',
                    'distance_metric': 'cosine'
                }
            }
        ]
})

index = SearchIndex(movie_schema, redis_client=client)
index.create()

In [ ]:
keys = index.load(
    df.to_dict(orient="records"),
    keys=[f"personalized_movie:{row_id}" for row_id in df.index],
)
movie_keys = dict(zip(df.index, keys))
print(f"Loaded {len(keys)} movies.")

## Add session personalization

On the first interaction, the session vector is the clicked movie's vector. On later interactions:

`session = beta * session + (1 - beta) * clicked_item`

With `beta=0.8`, each new interaction contributes 20% while the existing session retains 80%. We store the vector as float32 bytes with `SET ... EX 1800`, so Redis automatically removes inactive sessions.

At search time:

`search_vector = (1 - alpha) * query_embedding + alpha * session`

Start with `alpha=0.2`–`0.3`. `alpha=0` gives the original query, while `alpha=1` uses only the session. Without a session, return the query embedding unchanged. These functions accept lists or NumPy arrays and convert them to float32 before serializing or blending.

Use a distinct `user_id` per active session if a user can have several sessions. Inputs must use the same embedding model and dimensions; `alpha` and `beta` should be between 0 and 1.

In [ ]:
def update_session_vector(user_id, clicked_item_vec, redis_client, beta=0.8, ttl=1800):
    key = f"session:{user_id}:vec"
    clicked_item_vec = np.asarray(clicked_item_vec, dtype=np.float32)
    # shortcut: updates assume sequential events per session; use WATCH/MULTI for concurrent writers.
    existing = redis_client.get(key)
    if existing is not None:
        session_vec = np.frombuffer(existing, dtype=np.float32)
        new_vec = beta * session_vec + (1 - beta) * clicked_item_vec
    else:
        new_vec = clicked_item_vec
    redis_client.set(key, new_vec.tobytes(), ex=ttl)


def personalized_query_vec(user_id, query_embedding, redis_client, alpha=0.3):
    key = f"session:{user_id}:vec"
    query_embedding = np.asarray(query_embedding, dtype=np.float32)
    session_bytes = redis_client.get(key)
    if session_bytes is None:
        return query_embedding
    session_vec = np.frombuffer(session_bytes, dtype=np.float32)
    return (1 - alpha) * query_embedding + alpha * session_vec

### Quick correctness check

Use a temporary session to check cold start, initialization, EWMA, blending, and TTL. The small test vectors do not enter the movie index.

In [ ]:
check_user = f"check-{uuid4().hex}"
check_key = f"session:{check_user}:vec"
first = np.array([1.0, 0.0, 0.0], dtype=np.float32)
second = np.array([0.0, 1.0, 0.0], dtype=np.float32)

try:
    np.testing.assert_array_equal(personalized_query_vec(check_user, first, client), first)
    update_session_vector(check_user, first.tolist(), client)
    np.testing.assert_array_equal(np.frombuffer(client.get(check_key), dtype=np.float32), first)
    update_session_vector(check_user, second, client)
    expected = 0.8 * first + 0.2 * second
    np.testing.assert_allclose(np.frombuffer(client.get(check_key), dtype=np.float32), expected)
    np.testing.assert_allclose(
        personalized_query_vec(check_user, second, client), 0.7 * second + 0.3 * expected
    )
    np.testing.assert_array_equal(personalized_query_vec(check_user, second, client, alpha=0), second)
    np.testing.assert_allclose(personalized_query_vec(check_user, second, client, alpha=1), expected)
    assert 0 < client.ttl(check_key) <= 1800
finally:
    client.delete(check_key)

np.testing.assert_array_equal(personalized_query_vec(check_user, second, client), second)
print("Session personalization checks passed.")

## Search before any interactions

Start with a fresh demo session and a broad query. With no session vector, this is ordinary content search. The NumPy result is converted to a list for RedisVL's `VectorQuery` API.

In [ ]:
user_id = f"demo-{uuid4().hex}"
query_text = "A mysterious journey into the unknown"
query_embedding = np.asarray(vectorizer.embed(query_text), dtype=np.float32)

query = VectorQuery(
    vector=personalized_query_vec(user_id, query_embedding, client).tolist(),
    vector_field_name="embedding",
    num_results=5,
    return_fields=["title", "overview", "genres"],
)
baseline_results = index.query(query)
pd.DataFrame(baseline_results)

## Update the session when the user clicks or views movies

Simulate two interactions with horror movies. An application would receive the clicked movie's Redis key from its UI. Fetch only the stored embedding with `JSON.GET`; no model call is needed for an interaction.

In [ ]:
clicked_movies = df[df["genres"].apply(lambda genres: "Horror" in genres)].head(2)

for row_id, title in clicked_movies["title"].items():
    clicked_item_key = movie_keys[row_id]
    stored_vectors = client.json().get(clicked_item_key, "$.embedding")
    if not stored_vectors:
        raise ValueError(f"Movie embedding not found: {clicked_item_key}")
    update_session_vector(user_id, stored_vectors[0], client)
    print(f"Viewed: {title}")

print(f"Session expires in {client.ttl(f'session:{user_id}:vec')} seconds.")

## Search again with the same query

Keep the query text and search settings fixed. Only the blended vector changes. With `alpha=0.3`, the session contributes 30% and the query contributes 70%. Compare the titles below to see how recent horror views influence ranking. The exact results and how much they change depend on the query and interactions; different search vectors can still return the same top movies.

This is semantic vector search: blending biases the query's meaning, rather than adding a keyword filter. Existing RedisVL metadata filters can still be supplied through `filter_expression`.

In [ ]:
alpha = 0.3
query = VectorQuery(
    vector=personalized_query_vec(user_id, query_embedding, client, alpha=alpha).tolist(),
    vector_field_name="embedding",
    num_results=5,
    return_fields=["title", "overview", "genres"],
)
personalized_results = index.query(query)

pd.concat(
    {
        "Before interactions": pd.DataFrame(baseline_results)[["title", "vector_distance"]],
        "After interactions": pd.DataFrame(personalized_results)[["title", "vector_distance"]],
    },
    axis=1,
)

## Clean up

The session key expires automatically after 30 minutes without interactions. When you're finished, the cell below also removes it immediately and deletes this notebook's index and movie documents.

In [ ]:
client.delete(f"session:{user_id}:vec")
index.delete(drop=True)

You now have content filtering that adapts to recent interactions using two small functions and existing movie vectors. Try different queries, clicks, and values of `alpha`, starting around 0.2–0.3.

References: [RedisVL VectorQuery](https://docs.redisvl.com/en/latest/api/query.html#vectorquery) and [Redis JSON.GET](https://redis.io/docs/latest/commands/json.get/).